# Spatial-query review audit

This post-hoc audit reuses frozen training-selected queries.
The reference preserves each threshold mask size within an image
and independently randomizes their relative positions.
Counts are descriptive, not independent semantic evaluations.
Implementation: `src/lattmc/vision/spatial_review_codexgen.py`.

In [1]:
import json
from pathlib import Path
import numpy as np
import pandas as pd

root = Path.cwd()
folder = root / "vision_tokens/overcomplete/results"
folder = folder / "spatial_review"
audit = json.loads(
    (folder / "spatial_audit_codexgen.json").read_text())
rows = pd.DataFrame(audit["results"])
assert len(rows) == 420
assert audit["reference_exhaustive_cases"] == 284
rows.head()

,model,dataset,pair,features,threshold_multiplier,images,sites,pooled,common,independent_expected,separate_query_intersection
0,topk_k32_s0,imagenette,0,"[329, 507]",0.5,100,256,17,17,4.216983,17
1,topk_k32_s0,imagenette,0,"[329, 507]",1.0,100,256,4,4,0.268560,4
2,topk_k32_s0,imagenette,0,"[329, 507]",1.5,100,256,0,0,0.000000,0
3,topk_k32_s0,imagenette,1,"[1430, 871]",0.5,100,256,16,15,3.446247,15
4,topk_k32_s0,imagenette,1,"[1430, 871]",1.0,100,256,8,7,1.227788,7


## Every model and threshold

The pretrained RA queries become empty at multiplier 1.5.
A smaller discrepancy is not a model-quality ranking.

In [2]:
summary = pd.read_json(folder / "summary_codexgen.json")
summary

,model,multiplier,pooled,common,independent_expected,separate_query_intersection,discrepancy
0,topk_k32_s0,0.5,1084,1044,500.421800,1053,0.036900
1,topk_k32_s0,1.0,747,720,161.929031,740,0.036145
2,topk_k32_s0,1.5,168,140,15.044268,168,0.166667
3,batchtopk_k32_s0,0.5,537,456,201.789590,474,0.150838
4,batchtopk_k32_s0,1.0,263,131,44.998408,234,0.501901
5,batchtopk_k32_s0,1.5,34,16,2.362353,23,0.529412
6,jump_k32_s0,0.5,766,699,144.879778,708,0.087467
7,jump_k32_s0,1.0,282,216,28.651954,260,0.234043
8,jump_k32_s0,1.5,28,9,1.244046,18,0.678571
9,archetypal_k32_s0,0.5,495,344,269.223049,353,0.305051


## Per-dataset counts at the original thresholds

In [3]:
columns = ["pooled", "common", "independent_expected",
           "separate_query_intersection"]
fixed = rows[rows.threshold_multiplier == 1]
fixed.groupby(["model", "dataset"])[columns].sum()

pooled  common  independent_expected  \
model             dataset                                            
archetypal_k32_s0 dtd              1       0              0.125000   
                  imagenette      34      27             15.778921   
                  imagewoof       46      16             13.863286   
                  parts            8       6              3.418629   
                  pets           108      57             48.058467   
batchtopk_k32_s0  dtd              1       1              0.299783   
                  imagenette      61      45             11.320494   
                  imagewoof       73      39             12.136615   
                  parts           27      16              3.078581   
                  pets           101      30             18.162934   
jump_k32_s0       dtd              7       6              0.078125   
                  imagenette      44      35              4.978486   
                  imagewoof       94      71              8.672019   
                  parts           12      10              0.738246   
                  pets           125      94             14.185079   
pretrained_ra     dtd              0       0              0.000000   
                  imagenette      17      15             11.472916   
                  imagewoof        0       0              0.000000   
                  parts            1       1              0.408517   
                  pets             0       0              0.000000   
prisma_transcoder dtd            122      66             17.654667   
                  imagenette     115      60              8.308179   
                  imagewoof      124      71              9.395278   
                  parts          100      46             11.576809   
                  pets           303     159             25.907130   
relu_fixed_k32_s0 dtd            115      79             80.693512   
                  imagenette     125      61             53.561756   
                  imagewoof      178      86             80.960390   
                  parts          101      53             48.502810   
                  pets           382     194            204.781637   
topk_k32_s0       dtd              0       0              0.000000   
                  imagenette      29      27              5.663238   
                  imagewoof      212     203             40.466139   
                  parts           44      43              7.585654   
                  pets           462     447            108.214000   

                              separate_query_intersection  
model             dataset                                  
archetypal_k32_s0 dtd                                   0  
                  imagenette                           30  
                  imagewoof                            28  
                  parts                                 6  
                  pets                                 74  
batchtopk_k32_s0  dtd                                   1  
                  imagenette                           57  
                  imagewoof                            69  
                  parts                                20  
                  pets                                 87  
jump_k32_s0       dtd                                   7  
                  imagenette                           43  
                  imagewoof                            85  
                  parts                                11  
                  pets                                114  
pretrained_ra     dtd                                   0  
                  imagenette                           16  
                  imagewoof                             0  
                  parts                                 1  
                  pets                                  0  
prisma_transcoder dtd                                  69  
                  imagenette                           66  

## Independent consistency checks

The archived per-image arrays store pooled, common,
expected, and separate-query intersection values.
All original unscaled counts also matched prior results.

In [4]:
with np.load(folder / "per_image_codexgen.npz") as arrays:
    assert len(arrays.files) == len(rows)
    for row in audit["results"]:
        key = (f"{row['model']}_{row['dataset']}_"
               f"{row['pair']}_"
               f"{row['threshold_multiplier']}")
        values = arrays[key]
        assert np.all(values[:, 1] <= values[:, 3])
        assert np.all(values[:, 3] <= values[:, 0])
        for column, name in enumerate(columns):
            assert np.isclose(values[:, column].sum(),
                              row[name])
print("All 420 saved cases agree with per-image evidence.")

All 420 saved cases agree with per-image evidence.
